In [2]:
import pandas as pd
import numpy as np
import re
import torch

from bertopic import BERTopic
from collections import Counter

from topics_to_merge import topics_to_merge

In [21]:
topic_model = BERTopic.load("topic_models/251219")

In [251]:
#torch.cuda.empty_cache()

# recalculate embeddings for/and outlier reduction

In [6]:
docs = df["cleaned_content"].tolist()
# prepare df first (see below)

#embeddings = topic_model.embedding_model.embed(docs)

In [26]:
new_topics = topic_model.reduce_outliers(
    documents=docs,
    topics=topic_model.topics_,
    embeddings=embeddings,
    strategy="embeddings",
    threshold=0.015
)

In [67]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,233030,0_twitter_يا_status_じぇ,"[twitter, يا, status, じぇ, حافظ, يلعن, روحك, yo...",NaN
1,1,75723,1_music_apple_album_song,"[music, apple, album, song, bandcamp, iphone, ...",NaN
2,2,39595,2_mastodon_instance_social_fediverse,"[mastodon, instance, social, fediverse, v4, se...",NaN
3,3,41552,3_football_sport_f1_nfl,"[football, sport, f1, nfl, season, hockey, tea...",NaN
4,4,39010,4_climate_mac_usb_macos,"[climate, mac, usb, macos, ram, laptop, intel,...",NaN
...,...,...,...,...,...
357,357,1062,357_shave_wetshaving_razor_lather,"[shave, wetshaving, razor, lather, brush, sotd...",NaN
358,358,905,358_tennis_djokovic_racket_novak,"[tennis, djokovic, racket, novak, atp, wimbled...",NaN
359,359,11394,359_fuck_fucking_shit_pussy,"[fuck, fucking, shit, pussy, cock, hell, oh, h...",NaN
360,360,3447,360_98_197_76_88,"[98, 197, 76, 88, 395, cve, test, 1318, 51, 52]",NaN


In [29]:
topic_model.update_topics(docs,topics=new_topics)

2025-12-25 11:26:02,009 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [69]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,233030,0_twitter_يا_status_じぇ,"[twitter, يا, status, じぇ, حافظ, يلعن, روحك, yo...",NaN
1,1,75723,1_music_apple_album_song,"[music, apple, album, song, bandcamp, iphone, ...",NaN
2,2,39595,2_mastodon_instance_social_fediverse,"[mastodon, instance, social, fediverse, v4, se...",NaN
3,3,41552,3_football_sport_f1_nfl,"[football, sport, f1, nfl, season, hockey, tea...",NaN
4,4,39010,4_climate_mac_usb_macos,"[climate, mac, usb, macos, ram, laptop, intel,...",NaN
...,...,...,...,...,...
357,357,1062,357_shave_wetshaving_razor_lather,"[shave, wetshaving, razor, lather, brush, sotd...",NaN
358,358,905,358_tennis_djokovic_racket_novak,"[tennis, djokovic, racket, novak, atp, wimbled...",NaN
359,359,11394,359_fuck_fucking_shit_pussy,"[fuck, fucking, shit, pussy, cock, hell, oh, h...",NaN
360,360,3447,360_98_197_76_88,"[98, 197, 76, 88, 395, cve, test, 1318, 51, 52]",NaN


In [31]:
topic_model.save("topic_models/251219_reduced_outliers", serialization="safetensors", save_ctfidf=True, save_embedding_model="paraphrase-multilingual-MiniLM-L12-v2")

# prepare df

In [3]:
df = pd.read_csv("toots_per_instance_sampled_75_percentile.csv")

In [53]:
df.columns

Index(['instance', 'instance_id', 'cleaned_content'], dtype='object')

In [48]:
df.shape

(2814985, 5)

In [9]:
def strip_urls_and_handles(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+(@\w+)?", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [10]:
df["cleaned_content"] = (
    df["cleaned_content"]
    .apply(strip_urls_and_handles)
)

# merge topics

In [22]:
reduced_outliers_model = BERTopic.load("topic_models/251219_reduced_outliers")

In [246]:
reduced_outliers_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,233030,0_twitter_يا_status_じぇ,"[twitter, يا, status, じぇ, حافظ, يلعن, روحك, yo...",NaN
1,1,75723,1_music_apple_album_song,"[music, apple, album, song, bandcamp, iphone, ...",NaN
2,2,39595,2_mastodon_instance_social_fediverse,"[mastodon, instance, social, fediverse, v4, se...",NaN
3,3,41552,3_football_sport_f1_nfl,"[football, sport, f1, nfl, season, hockey, tea...",NaN
4,4,39010,4_climate_mac_usb_macos,"[climate, mac, usb, macos, ram, laptop, intel,...",NaN
...,...,...,...,...,...
357,357,1062,357_shave_wetshaving_razor_lather,"[shave, wetshaving, razor, lather, brush, sotd...",NaN
358,358,905,358_tennis_djokovic_racket_novak,"[tennis, djokovic, racket, novak, atp, wimbled...",NaN
359,359,11394,359_fuck_fucking_shit_pussy,"[fuck, fucking, shit, pussy, cock, hell, oh, h...",NaN
360,360,3447,360_98_197_76_88,"[98, 197, 76, 88, 395, cve, test, 1318, 51, 52]",NaN


## check that all topics are categorized

In [247]:
info = reduced_outliers_model.get_topic_info().set_index("Topic")
topic_labels = dict(zip(info.index, info["Name"]))

for i, group in enumerate(topics_to_merge, start=1):
    print(f"group {i}:")
    for tid in group:
        label = topic_labels.get(tid, "<UNKNOWN>")
        print(f"{label}")
    print() 

group 1:
0_twitter_يا_status_じぇ
2_mastodon_instance_social_fediverse
7_youtube_watch_twitch_video
51_blog_internet_web_post
53_musk_elon_twitter_elonmusk
75_threads_thread_activitypub_meta
92_account_accounts_instance_my
110_bluesky_bsky_invite_blue
115_facebook_meta_instagram_fb
140_reddit_irc_stopdronebl_liberachat
168_fediverse_welcome_fediverso_observer
174_friends_friend_friendship_friendica
180_post_posts_posting_here
195_fedi_fedora_fedilab_fediblock
201_pleroma_pr_shc_kanagawa_jp_clojure_instance
203_rss_feed_feeds_reader
230_instagram_account_photos_image
233_pixelfed_pixel_image_images
238_reddit_exmormon_fade_comments
246_whatsapp_signal_telegram_messenger
248_ivory_tapbots_tweetbot_app
256_hashtags_hashtag_span_tags
262_johnmastodon_funfacts_john_mastodon
308_followers_fandoms_follow_follower
342_blue_imastodon_bluey_blaue

group 2:
1_music_apple_album_song
43_podcast_podcasts_audio_episode
131_dance_dancing_ballet_dances
156_eurovision_eurovision2023_x27_esc
197_spotify_pl

## actually merge topics

In [248]:
reduced_outliers_model.merge_topics(docs, topics_to_merge)

In [252]:
reduced_outliers_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,414066,0_mastodon_to_39_twitter,"[mastodon, to, 39, twitter, com, the, it, and,...",[This is my first post on Mastodon. I have no ...
1,1,236531,1_39_to_the_and,"[39, to, the, and, you, it, of, is, in, my]",[If you know what you are doing 3/4 of the tim...
2,2,153861,2_the_in_to_and,"[the, in, to, and, of, quot, de, 2023, 39, it]",[The time now is: Australia/Sydney: 2023-11-13...
3,3,148550,3_to_the_and_39,"[to, the, and, 39, it, for, of, is, in, that]",[Glad to see more Rhapsody coverage. When I wa...
4,4,144579,4_de_in_the_der,"[de, in, the, der, quot, of, to, la, 39, die]","[Why Waterford city? Well, I&#39;m from Waterf..."
5,5,130588,5_to_the_it_39,"[to, the, it, 39, and, is, for, on, my, linux]",[Going forward my direct contact lines will be...
6,6,123786,6_to_and_the_39,"[to, and, the, 39, of, my, in, it, that, is]",[@ DarcMoughty It depends on how you define it...
7,7,121603,7_die_quot_zøt_akkoma,"[die, quot, zøt, akkoma, e5, the, natty, 39, e...",[保安局指美國制裁釀資安漏洞 擬斥1.9億換人事紀錄系統 inmediahk.net/nod...
8,8,118029,8_the_of_and_39,"[the, of, and, 39, to, it, in, is, quot, that]",[@ libreture This is nice. My problem with Goo...
9,9,109368,9_music_the_apple_com,"[music, the, apple, com, 39, to, and, of, on, ...",[Now playing on Mixfix Radio: backyardigans do...


In [237]:
reduced_outliers_model.save("topic_models/251219_reduced_outliers_merged_topics", serialization="safetensors", save_ctfidf=True, save_embedding_model="paraphrase-multilingual-MiniLM-L12-v2")

# transform docs with merged model to calculate new topics & probs per item in docs

In [4]:
reduced_outliers_merged_model = BERTopic.load("topic_models/251219_reduced_outliers_merged_topics")

In [8]:
merged_topics, merged_probs = reduced_outliers_merged_model.transform(docs)

In [9]:
merged_probs

array([0.22341305, 0.50430024, 0.6066242 , ..., 0.2502829 , 0.31721622,
       0.3874546 ], dtype=float32)

In [10]:
df["topic"] = merged_topics 
df["topic_prob"] = merged_probs.max()

In [11]:
reduced_outliers_merged_model.get_topics()

{0: [['mastodon', 0.02813834050417026],
  ['to', 0.02209875650144803],
  ['39', 0.02133875598733121],
  ['twitter', 0.021222892512136913],
  ['com', 0.020443318387746785],
  ['the', 0.01877088923349335],
  ['it', 0.016149087837199972],
  ['and', 0.015765332521955774],
  ['on', 0.015567063241212271],
  ['youtube', 0.01534117745116656]],
 1: [['39', 0.021974710192686497],
  ['to', 0.021825483126645562],
  ['the', 0.021552434677047667],
  ['and', 0.019137646103760442],
  ['you', 0.01743902373672628],
  ['it', 0.01722480509763552],
  ['of', 0.016375636352700434],
  ['is', 0.015026040556018153],
  ['in', 0.014819706231184577],
  ['my', 0.014674335422366151]],
 2: [['the', 0.02215849162674519],
  ['in', 0.017805257121377687],
  ['to', 0.01583532394831263],
  ['and', 0.015230187311926324],
  ['of', 0.015226625026370845],
  ['quot', 0.014830180202008547],
  ['de', 0.014313165788512468],
  ['2023', 0.01420120608723962],
  ['39', 0.01367891472318234],
  ['it', 0.013143919952622348]],
 3: [['to',

In [12]:
reduced_outliers_merged_model.set_topic_labels({0: "social_media_plaforms_communities", 
 1: "life_relationship",
 2: "environment_climate_weather",
 3: "software_programming",
 4: "countries_local_contexts",
 5: "hardware_devices_infrastructure",
 6: "health_mental_health_body",
 7: "misc",
 8: "books_movies_fandom",
 9: "music", 
 10: "arts_crafts_design", 
 11: "current_political_events",
 12: "science_research",
 13: "travel_mobility_transport",
 14: "food_drinks",
 15: "identity_discrimination_subculture",
 16: "economy_finance", 
 17: "cloud_networks_security_privacy",
 18: "animals_nature",
 19: "news_politics",
 20: "games_gaming",
 21: "sports",
 22: "humor_memes_play",
 23: "religion",
 24: "ai_silicon_valley",
 25: "adult_content",
})

In [13]:
reduced_outliers_merged_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,0,414066,0_mastodon_to_39_twitter,social_media_plaforms_communities,"[mastodon, to, 39, twitter, com, the, it, and,...",NaN
1,1,236531,1_39_to_the_and,life_relationship,"[39, to, the, and, you, it, of, is, in, my]",NaN
2,2,153861,2_the_in_to_and,environment_climate_weather,"[the, in, to, and, of, quot, de, 2023, 39, it]",NaN
3,3,148550,3_to_the_and_39,software_programming,"[to, the, and, 39, it, for, of, is, in, that]",NaN
4,4,144579,4_de_in_the_der,countries_local_contexts,"[de, in, the, der, quot, of, to, la, 39, die]",NaN
5,5,130588,5_to_the_it_39,hardware_devices_infrastructure,"[to, the, it, 39, and, is, for, on, my, linux]",NaN
6,6,123786,6_to_and_the_39,health_mental_health_body,"[to, and, the, 39, of, my, in, it, that, is]",NaN
7,7,121603,7_die_quot_zøt_akkoma,misc,"[die, quot, zøt, akkoma, e5, the, natty, 39, e...",NaN
8,8,118029,8_the_of_and_39,books_movies_fandom,"[the, of, and, 39, to, it, in, is, quot, that]",NaN
9,9,109368,9_music_the_apple_com,music,"[music, the, apple, com, 39, to, and, of, on, ...",NaN


# n top topics per server

In [14]:
def calculate_top_n(df, topic_model, n=10): 

    server_topic_weighted = (
        df.groupby(["instance", "topic"])["topic_prob"]
        .sum()
        .reset_index(name="weighted_prob_sum")
    )
    
    top_topics_per_server = (
        server_topic_weighted.sort_values(["instance", "weighted_prob_sum"], ascending=[True, False])
        .groupby("instance")
        .head(n)
    )
    
    topic_info_df = topic_model.get_topic_info()[["Topic", "CustomName"]]
    topic_info_dict = dict(zip(topic_info_df["Topic"], topic_info_df["CustomName"]))
    top_topics_per_server["topic_name"] = top_topics_per_server["topic"].map(topic_info_dict)
    
    server_total = top_topics_per_server.groupby("instance")["weighted_prob_sum"].transform("sum")
    top_topics_per_server["relative_share"] = top_topics_per_server["weighted_prob_sum"] / server_total
    
    
    server_topics_dict = (
        top_topics_per_server.groupby("instance")[["topic_name", "relative_share"]]
        .apply(lambda x: list(zip(x["topic_name"], x["relative_share"])))
        .to_dict()
    )

    return server_topics_dict

In [15]:
server_topics_dict = calculate_top_n(df, reduced_outliers_merged_model, n=26)

In [16]:
server_topics_dict

{'0.0n.ee': [('ai_silicon_valley', 0.19298245012760162),
  ('social_media_plaforms_communities', 0.17543859779834747),
  ('cloud_networks_security_privacy', 0.14035087823867798),
  ('misc', 0.08771929889917374),
  ('humor_memes_play', 0.08771929889917374),
  ('countries_local_contexts', 0.07017543911933899),
  ('software_programming', 0.035087719559669495),
  ('hardware_devices_infrastructure', 0.035087719559669495),
  ('food_drinks', 0.035087719559669495),
  ('health_mental_health_body', 0.017543859779834747),
  ('music', 0.017543859779834747),
  ('current_political_events', 0.017543859779834747),
  ('science_research', 0.017543859779834747),
  ('identity_discrimination_subculture', 0.017543859779834747),
  ('economy_finance', 0.017543859779834747),
  ('animals_nature', 0.017543859779834747),
  ('religion', 0.017543859779834747)],
 '000000420.xyz': [('social_media_plaforms_communities', 0.3749999701976776),
  ('health_mental_health_body', 0.125),
  ('food_drinks', 0.125),
  ('economy_

In [17]:
all_instances = df["instance"].unique()
all_topic_names = set(topic for topics in server_topics_dict.values() for topic, _ in topics)

rows = []
for instance in all_instances:
    row = {"instance": instance}
    topic_dict = dict(server_topics_dict.get(instance, []))
    
    for topic_name in all_topic_names:
        row[topic_name] = topic_dict.get(topic_name, 0)
    
    rows.append(row)

server_topic_df = pd.DataFrame(rows)

In [18]:
server_topic_df.columns

Index(['instance', 'sports', 'animals_nature', 'science_research',
       'hardware_devices_infrastructure', 'cloud_networks_security_privacy',
       'news_politics', 'life_relationship', 'adult_content', 'food_drinks',
       'current_political_events', 'environment_climate_weather',
       'arts_crafts_design', 'humor_memes_play', 'countries_local_contexts',
       'identity_discrimination_subculture', 'travel_mobility_transport',
       'health_mental_health_body', 'software_programming',
       'social_media_plaforms_communities', 'religion', 'economy_finance',
       'games_gaming', 'ai_silicon_valley', 'misc', 'books_movies_fandom',
       'music'],
      dtype='object')

In [19]:
server_topic_df.to_csv("community_type_251219_reduced_merged.csv", index=False)

### get topics that cover 95 percent of all toots per server

In [20]:
def calculate_95_percent(df, topic_model, threshold=0.95):

    server_topic_weighted = (
        df.groupby(["instance", "topic"])["topic_prob"]
        .sum()
        .reset_index(name="weighted_prob_sum")
    )

    server_topic_weighted["relative_share"] = (
        server_topic_weighted
        .groupby("instance")["weighted_prob_sum"]
        .transform(lambda x: x / x.sum())
    )

    server_topic_weighted = server_topic_weighted.sort_values(
        ["instance", "relative_share"],
        ascending=[True, False]
    )


    rows = []

    for instance, group in server_topic_weighted.groupby("instance"):
        total = 0.0

        for _, row in group.iterrows():
            rows.append(row)
            total += row["relative_share"]

            if total >= threshold:
                break

    top_95_topics_per_server = pd.DataFrame(rows)

    topic_info_df = topic_model.get_topic_info()[["Topic", "CustomName"]]
    topic_info_dict = dict(zip(topic_info_df["Topic"], topic_info_df["CustomName"]))

    top_95_topics_per_server["topic_name"] = (
        top_95_topics_per_server["topic"].map(topic_info_dict)
    )

    return top_95_topics_per_server


In [23]:
top_95_df = calculate_95_percent(df, reduced_outliers_model)

KeyError: "['CustomName'] not in index"

In [ ]:
top_95_df["instance"].value_counts()

In [ ]:
top_95_df[top_95_df["instance"] == "mastodon.social"]

# cluster topics

In [24]:
topic_cols = [c for c in server_topic_df.columns if c != "instance"]

topic_means = server_topic_df[topic_cols].mean().sort_values(ascending=False)

In [25]:
topic_means

social_media_plaforms_communities     0.188215
cloud_networks_security_privacy       0.091646
humor_memes_play                      0.072780
hardware_devices_infrastructure       0.056315
arts_crafts_design                    0.039259
life_relationship                     0.038680
misc                                  0.037178
current_political_events              0.037003
software_programming                  0.035935
animals_nature                        0.035107
music                                 0.033305
ai_silicon_valley                     0.031214
environment_climate_weather           0.030053
food_drinks                           0.029272
travel_mobility_transport             0.027474
games_gaming                          0.026853
countries_local_contexts              0.025481
health_mental_health_body             0.025407
books_movies_fandom                   0.024822
adult_content                         0.021001
economy_finance                       0.020057
news_politics

# Verteilung der Topics

In [45]:
#server_topic_df = pd.read_csv("community_type_251219_reduced_merged.csv.csv")

# 'instance' ist die ID-Spalte, alles andere sind Topics
topic_cols = server_topic_df.columns.drop("instance")

# 1) Gesamte Präsenz jedes Topics (über alle Server aufsummiert)
topic_total = server_topic_df[topic_cols].sum()

# 2) Durchschnittlicher Anteil pro Server
topic_mean = server_topic_df[topic_cols].mean()

# 3) Als DataFrame zusammenfassen
topic_summary = pd.DataFrame({
    "total_weight": topic_total,
    "mean_share": topic_mean
}).sort_values("total_weight", ascending=False)

print(topic_summary)
print(server_topic_df[topic_cols].sum(axis=1).describe())


                                    total_weight  mean_share
social_media_plaforms_communities    1907.939559    0.188215
cloud_networks_security_privacy       929.013044    0.091646
humor_memes_play                      737.773656    0.072780
hardware_devices_infrastructure       570.868929    0.056315
arts_crafts_design                    397.965404    0.039259
life_relationship                     392.095445    0.038680
misc                                  376.868420    0.037178
current_political_events              375.095872    0.037003
software_programming                  364.270814    0.035935
animals_nature                        355.875933    0.035107
music                                 337.613527    0.033305
ai_silicon_valley                     316.415000    0.031214
environment_climate_weather           304.648914    0.030053
food_drinks                           296.734104    0.029272
travel_mobility_transport             278.499881    0.027474
games_gaming            

In [27]:
topic_groups = {

    # Culture, Media & Entertainment
    "books_movies_fandom": "Culture, Media & Entertainment",
    "music": "Culture, Media & Entertainment",
    "arts_crafts_design": "Culture, Media & Entertainment",
    "sports": "Culture, Media & Entertainment",
    "humor_memes_play": "Culture, Media & Entertainment",
    "animals_nature": "Culture, Media & Entertainment",
    "food_drinks": "Culture, Media & Entertainment",
    "games_gaming": "Culture, Media & Entertainment",

    # Technology
    "cloud_networks_security_privacy": "Technology",
    "hardware_devices_infrastructure": "Technology",
    "software_programming": "Technology",
    "social_media_plaforms_communities": "Technology",

    # Politics, Economy & Power
    "news_politics": "Politics, Economy & Power",
    "current_political_events": "Politics, Economy & Power",
    "ai_silicon_valley": "Politics, Economy & Power",
    "countries_local_contexts": "Politics, Economy & Power",
    "economy_finance": "Politics, Economy & Power",

    # Society, Identity and Everyday Life
    "identity_discrimination_subculture": "Society, Identity and Everyday Life",
    "life_relationship": "Society, Identity and Everyday Life",
    "adult_content": "Society, Identity and Everyday Life",
    "religion": "Society, Identity and Everyday Life",


    # Science
    "environment_climate_weather": "Science",
    "science_research": "Science",
    "health_mental_health_body": "Science",

    # Other
    "travel_mobility_transport": "Other",
    "misc": "Other"
}

In [28]:
topic_summary = topic_summary.reset_index().rename(columns={"index": "Topic"})

topic_summary["Group"] = topic_summary["Topic"].map(topic_groups)

In [29]:
topic_summary[topic_summary["Group"].isna()]

,Topic,total_weight,mean_share,Group


In [30]:
group_summary = (
    topic_summary
        .groupby("Group")["mean_share"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
)

In [31]:
group_summary

,Group,mean_share
0,Technology,0.372111
1,"Culture, Media & Entertainment",0.276918
2,"Politics, Economy & Power",0.131689
3,"Society, Identity and Everyday Life",0.084614
4,Science,0.070016
5,Other,0.064651


## weighted

In [40]:
# 1. get weighted topic means (your code)
server_sizes = df.groupby("instance").size()
topic_means_weighted = (
    server_topic_df.set_index("instance")[topic_cols]
    .multiply(server_sizes, axis=0)
    .sum() / server_sizes.sum()
)

# 2. convert to dataframe to merge with group labels
topic_means_weighted_df = topic_means_weighted.reset_index()
topic_means_weighted_df.columns = ["topic_name", "mean_share_weighted"]

topic_summary_weighted = topic_means_weighted_df.merge(
    topic_summary[["Topic", "Group"]].drop_duplicates(),
    left_on="topic_name",
    right_on="Topic",
    how="left"
)

group_summary_weighted = (
    topic_summary_weighted
    .groupby("Group")["mean_share_weighted"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

comparison = group_summary.merge(
    group_summary_weighted,
    on="Group"
).rename(columns={"mean_share": "unweighted", "mean_share_weighted": "weighted"})
comparison["difference"] = comparison["weighted"] - comparison["unweighted"]
print(comparison.sort_values("difference", ascending=False))

0          0.0n.ee
1          0.0n.ee
2          0.0n.ee
3          0.0n.ee
4          0.0n.ee
            ...   
2814980    ê©°.com
2814981    ê©°.com
2814982    ê©°.com
2814983    ê©°.com
2814984    ê©°.com
Name: instance, Length: 2814985, dtype: object
instance
0.0n.ee              57
000000420.xyz         8
0rb.it                4
0v0.ch                9
0verti.me           102
                   ... 
zyl.se                1
zzz.grtsk.com        37
Î¼.computer         512
Õ©.Õ¹Õ¸Õ¬.Õ°Õ¡Õµ    752
ê©°.com              20
Length: 10137, dtype: int64
                                 Group  unweighted  weighted  difference
2            Politics, Economy & Power    0.131689  0.162008    0.030318
1       Culture, Media & Entertainment    0.276918  0.298782    0.021864
4                              Science    0.070016  0.083458    0.013441
3  Society, Identity and Everyday Life    0.084614  0.091510    0.006896
5                                Other    0.064651  0.060294   -0.004357
0    

In [44]:
print(df['instance'])
print(server_sizes)
print(df.head())

0          0.0n.ee
1          0.0n.ee
2          0.0n.ee
3          0.0n.ee
4          0.0n.ee
            ...   
2814980    ê©°.com
2814981    ê©°.com
2814982    ê©°.com
2814983    ê©°.com
2814984    ê©°.com
Name: instance, Length: 2814985, dtype: object
instance
0.0n.ee              57
000000420.xyz         8
0rb.it                4
0v0.ch                9
0verti.me           102
                   ... 
zyl.se                1
zzz.grtsk.com        37
Î¼.computer         512
Õ©.Õ¹Õ¸Õ¬.Õ°Õ¡Õµ    752
ê©°.com              20
Length: 10137, dtype: int64
  instance                 instance_id  \
0  0.0n.ee  0.0n.ee_Ac9HuDQuvOCyoWaAoS   
1  0.0n.ee  0.0n.ee_Ac6BeNffuTJsjZfhjc   
2  0.0n.ee  0.0n.ee_Abw487pqbrYh5qVExM   
3  0.0n.ee  0.0n.ee_Abw3z52XdJNNol7xM8   
4  0.0n.ee  0.0n.ee_AbtjcJwxe47G05T2Ce   

                                     cleaned_content  topic  topic_prob  
0  발키리기체로 뽑는 발제네는 개당 300엔, 인터넷으로 뽑는 프리미엄 발제네는 500...     16    0.967858  
1  @ ai Hey Siri, please make me an あだ名. h

In [36]:
# normalize weighted shares to sum to 1
topic_means_weighted_df["weighted_share_normalized"] = (
    topic_means_weighted_df["mean_share_weighted"] / topic_means_weighted_df["mean_share_weighted"].sum()
)

# merge group labels and sort
topic_weighted_26 = topic_means_weighted_df.merge(
    topic_summary[["Topic", "Group"]].drop_duplicates(),
    left_on="topic_name",
    right_on="Topic",
    how="left"
)[["topic_name", "Group", "weighted_share_normalized"]].sort_values(
    "weighted_share_normalized", ascending=False
)

print(topic_weighted_26.to_string())


                            topic_name                                Group  weighted_share_normalized
18   social_media_plaforms_communities                           Technology                   0.160385
4      cloud_networks_security_privacy                           Technology                   0.074576
12                    humor_memes_play       Culture, Media & Entertainment                   0.073076
9             current_political_events            Politics, Economy & Power                   0.051246
3      hardware_devices_infrastructure                           Technology                   0.042270
11                  arts_crafts_design       Culture, Media & Entertainment                   0.039129
25                               music       Culture, Media & Entertainment                   0.037897
1                       animals_nature       Culture, Media & Entertainment                   0.037210
10         environment_climate_weather                              Scien